# Grant OTel Simulator App Permissions

Grants the Databricks App service principal access to the OTel UC tables.
Run this notebook after deploying the `otel-simulator` app.

In [ ]:
# --- Configuration ---
APP_NAME = "otel-simulator"
CATALOG = "bx3"
SCHEMA = "otel_demo"
TABLES = [
    f"{CATALOG}.{SCHEMA}.otel_spans_v2",
    f"{CATALOG}.{SCHEMA}.otel_logs_v2",
    f"{CATALOG}.{SCHEMA}.otel_metrics",
]

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
app = w.apps.get(APP_NAME)
sp_app_id = app.service_principal_client_id
print(f"App: {app.name}")
print(f"Service Principal: {app.service_principal_name}")
print(f"Application ID:   {sp_app_id}")

In [ ]:
grants = [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{sp_app_id}`",
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{sp_app_id}`",
] + [
    f"GRANT MODIFY, SELECT ON TABLE {t} TO `{sp_app_id}`"
    for t in TABLES
]

print("Applying grants:\n")
for stmt in grants:
    print(f"  {stmt}")
    spark.sql(stmt)

print("\nAll grants applied successfully.")

In [ ]:
# Verify grants
for table in TABLES:
    display(spark.sql(f"SHOW GRANTS ON TABLE {table}"))